# 📊 Sales Performance EDA
**Author:** Vanil B. Patel | KSV University

End-to-end Exploratory Data Analysis on 5-year retail sales data (50K+ records)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('✅ Libraries ready!')

In [ ]:
# Generate synthetic 5-year retail sales data
np.random.seed(42)
n = 50000
categories = ['Electronics', 'Clothing', 'Groceries', 'Furniture', 'Sports', 'Books']
regions = ['North', 'South', 'East', 'West']
cat_weights = [0.28, 0.22, 0.18, 0.14, 0.11, 0.07]

dates = pd.date_range('2020-01-01', '2024-12-31', periods=n)
df = pd.DataFrame({
    'date': dates,
    'category': np.random.choice(categories, n, p=cat_weights),
    'region': np.random.choice(regions, n),
    'units': np.random.randint(1, 50, n),
    'unit_price': np.random.uniform(10, 500, n).round(2),
    'discount_pct': np.random.choice([0, 5, 10, 15, 20], n, p=[0.4, 0.2, 0.2, 0.1, 0.1]),
})
df['revenue'] = (df['units'] * df['unit_price'] * (1 - df['discount_pct']/100)).round(2)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['month_name'] = df['date'].dt.strftime('%b')

print(f'Dataset: {df.shape}')
print(f'Total Revenue: ₹{df.revenue.sum():,.0f}')
df.head()

In [ ]:
# ── Revenue by Category ──
cat_rev = df.groupby('category')['revenue'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cat_rev.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Total Revenue by Category', fontsize=13, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Revenue (₹)')
axes[0].tick_params(axis='x', rotation=30)

axes[1].pie(cat_rev, labels=cat_rev.index, autopct='%1.1f%%',
            colors=sns.color_palette('Blues_r', len(cat_rev)))
axes[1].set_title('Revenue Share by Category', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('revenue_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Monthly Revenue Trend (5 years) ──
monthly = df.groupby(['year', 'month'])['revenue'].sum().reset_index()
monthly['date'] = pd.to_datetime(monthly[['year', 'month']].assign(day=1))

plt.figure(figsize=(14, 5))
for yr in sorted(monthly['year'].unique()):
    data = monthly[monthly['year'] == yr]
    plt.plot(data['month'], data['revenue'], marker='o', label=str(yr), linewidth=2)

plt.title('Monthly Revenue Trend by Year', fontsize=13, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue (₹)')
plt.xticks(range(1,13), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.legend(title='Year')
plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Regional Sales Heatmap ──
pivot = df.pivot_table(values='revenue', index='region', columns='quarter', aggfunc='sum')
pivot.columns = ['Q1', 'Q2', 'Q3', 'Q4']

plt.figure(figsize=(8, 4))
sns.heatmap(pivot/1e6, annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, cbar_kws={'label': 'Revenue (₹M)'})
plt.title('Regional Revenue Heatmap by Quarter (₹M)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('regional_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── KPI Summary ──
print('=' * 45)
print('           📊 SALES KPI SUMMARY')
print('=' * 45)
print(f'  Total Revenue       : ₹{df.revenue.sum()/1e6:.1f}M')
print(f'  Total Orders        : {len(df):,}')
print(f'  Avg Order Value     : ₹{df.revenue.mean():.0f}')
print(f'  Top Category        : {cat_rev.index[0]} ({cat_rev.iloc[0]/df.revenue.sum():.0%})')
print(f'  Best Quarter        : Q{df.groupby("quarter")["revenue"].sum().idxmax()}')
print(f'  Best Region         : {df.groupby("region")["revenue"].sum().idxmax()}')
print(f'  Avg Discount        : {df.discount_pct.mean():.1f}%')
print('=' * 45)